# Homework: Building a Wikipedia Agent

In this homework we'll build an agent that can search Wikipedia and fetch page content to answer questions.

We'll start by defining tools, then create an agent that uses these tools to answer questions about any topic.

Note: For all questions, if your answer doesn't match exactly due to differences in tools or environments, pick the closest option.


## Wikipedia API Setup

We'll use two Wikipedia API endpoints.

**Search API** - Find pages related to a topic:

```text
https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=YOUR_QUERY
```

Replace `YOUR_QUERY` with your search term. Use `+` for spaces (for example, `lesser+capybara`).

**Get Page API** - Fetch the raw content of a page:

```text
https://en.wikipedia.org/w/index.php?title=PAGE_TITLE&action=raw
```

Replace `PAGE_TITLE` with the exact title of the Wikipedia page.

Wikipedia requires a `User-Agent` header on API requests. Without it, your requests may be blocked. Pass it in every `requests.get` call:


In [ ]:
import requests

headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
response = requests.get(
    "https://en.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": "lesser capybara",
    },
    headers=headers,
)
response.raise_for_status()
response.json()

You can test these APIs directly in your browser to see how they work.

Quick check of the Get Page API:


In [ ]:
page_response = requests.get(
    "https://en.wikipedia.org/w/index.php",
    params={
        "title": "Capybara",
        "action": "raw",
    },
    headers=headers,
)
page_response.raise_for_status()
page_response.text[:500]

## Question 1. Define the Search Tool

Create a `search_wikipedia` function that:

- Takes a query string as input
- Calls the Wikipedia Search API
- Returns the search results

Test your function with the query `"capybara"`.

How many total results did the search return?


In [ ]:
import requests

HEADERS = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}


def search_wikipedia(query: str) -> dict:
    """Search Wikipedia and return the raw API response query payload."""
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers=HEADERS,
    )
    response.raise_for_status()
    return response.json()["query"]


results = search_wikipedia("capybara")
results["searchinfo"]

In [ ]:
total_results = results["searchinfo"]["totalhits"]
len(results["search"]), total_results

## Question 2. Analyzing Search Results

Using your `search_wikipedia` function, search for `"capybara"` again.

Look at the titles in the search results. How many of the results contain the word `"capybara"` (case-insensitive) in their title?


Hint: Check each result's `title` field.


In [6]:
results = search_wikipedia("capybara")
titles = [item["title"] for item in results["search"]]
matches = [title for title in titles if "capybara" in title.lower()]

for title in titles:
    print(title)

len(matches)

Capybara
Lesser capybara
Capybara (disambiguation)
Capybara Party
Capybara Games
Capybara (software)
Hydrochoerus
Caviidae
Kyle Thomas (influencer)
Flow (2024 film)


6

## Question 3. Define the Get Page Tool

Create a `get_page` function that:

- Takes a page title as input
- Calls the Wikipedia Get Page API
- Returns the page content as a string

Test your function by fetching the page titled `"Capybara"`.

How many characters are in the page content?

Hint: Use `len(content)` to count characters.

Note: Wikipedia pages can be long. If you're using a smaller or free model with a limited context window, you may need to chunk the page content and only send the most relevant chunk to the LLM. The right chunk size depends on your model's input token limit.


In [ ]:
def get_page(title: str) -> str:
    """Fetch the raw Wikipedia page content for a given title."""
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={
            "title": title,
            "action": "raw",
        },
        headers=HEADERS,
    )
    response.raise_for_status()
    return response.text


content = get_page("Capybara")
len(content)

## Question 4. Agent Setup

Now create an agent with access to both tools:

- `search_wikipedia` - Search for Wikipedia pages
- `get_page` - Fetch the content of a specific page

You can use any framework (ToyAIKit, PydanticAI, OpenAI Agents SDK, LangChain, etc.) or implement the agentic loop yourself from scratch.

You can also use any LLM provider (OpenAI, Anthropic, AWS Bedrock, etc.).

In your submission, please specify:

- Which framework did you choose (or "custom" if implementing from scratch)?
- Which LLM provider did you use?


In [17]:
import nest_asyncio
from dotenv import load_dotenv
from pydantic_ai import Agent

nest_asyncio.apply()
load_dotenv()


def search_wikipedia(query: str) -> list[dict]:
    """
    Search Wikipedia for pages matching a query.

    Args:
        query: The search query string.

    Returns:
        A list of search result dicts with title, snippet, and pageid.
    """
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers=HEADERS,
    )
    response.raise_for_status()
    return response.json()["query"]["search"]


def get_page(title: str) -> str:
    """
    Fetch the raw Wikipedia page content for a given title.

    Args:
        title: Exact Wikipedia page title.

    Returns:
        Page wikitext content (truncated for model context).
    """
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={
            "title": title,
            "action": "raw",
        },
        headers=HEADERS,
    )
    response.raise_for_status()
    content = response.text
    # Keep tool output within a reasonable context budget
    max_chars = 12000
    if len(content) > max_chars:
        return content[:max_chars] + "\n\n[truncated]"
    return content


instructions = """
You are a Wikipedia research assistant.

Use search_wikipedia to find relevant pages, then get_page to read
the most useful page(s) before answering.

Prefer exact page titles returned by search.
Answer using only information from the retrieved Wikipedia content.
"""

wikipedia_agent = Agent(
    name="wikipedia",
    model="openai:gpt-4o-mini",
    instructions=instructions,
    tools=[search_wikipedia, get_page],
)

## Question 5. Testing Your Agent - Single Page

Use your agent to answer this question:

`"What is this page about? https://en.wikipedia.org/wiki/Capybara"`

Your agent should:

- Use the `get_page` tool to fetch the Capybara page
- Provide a concise summary of what the page is about

What is the main topic of the Capybara page according to your agent?

Keep the answer ready for submission.


In [10]:
result = wikipedia_agent.run_sync(
    "What is this page about? https://en.wikipedia.org/wiki/Capybara"
)
print(result.output)
result.usage

The Wikipedia page on **Capybara** provides detailed information about this species, scientifically known as *Hydrochoerus hydrochaeris*, which is recognized as the largest rodent currently living. The capybara is indigenous to most of South America, except for Chile, and is characterized as a semi-aquatic herbivore typically residing in savannas and dense forests near freshwater bodies.

Key highlights from the page include:

- **Habitat and Diet**: Capybaras primarily feed on grasses and aquatic plants and are very social animals, often living in groups ranging from 10 to 20, but can aggregate in larger groups.

- **Physical Description**: They have a barrel-shaped body, broad head, and coarse fur, varying in color from reddish-brown on top to yellowish-brown underneath. Adults typically weigh between 35 to 66 kg.

- **Etymology**: The name "capybara" comes from the Tupi language, meaning "one who eats slender leaves". The genus name derives from Greek, referring to their aquatic nat

RunUsage(cost=Decimal('0.00080850'), details={'reasoning_tokens': 0}, input_tokens=4002, output_tokens=347, requests=2, tool_calls=1)

## Question 6. Test The Agent - Search Then Fetch

Next, test if your agent can use both tools together.

Ask your agent:

`"What are the main threats to capybara populations?"`

Your agent should:

- First use `search_wikipedia` to find pages about capybara threats
- Then use `get_page` to fetch the relevant pages and extract specific threat information

How many total tool calls (search + fetch combined) did your agent make?

In your answer, provide:

- Which tools it used and in what order
- The final answer your agent gave


In [ ]:
from pydantic_ai.messages import FunctionToolCallEvent


class NamedCallback:
    def __init__(self, agent):
        self.agent_name = agent.name
        self.calls = []

    async def print_function_calls(self, ctx, event):
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            self.calls.append((tool_name, args))
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)


callback = NamedCallback(wikipedia_agent)
result = wikipedia_agent.run_sync(
    "What are the main threats to capybara populations?",
    event_stream_handler=callback,
)

print("\nTotal tool calls:", len(callback.calls))
for i, (name, args) in enumerate(callback.calls, 1):
    print(i, name, args)

print("\nFinal answer:\n")
print(result.output)